# Phase 3 — Feature Engineering
**UCI Hydraulic Systems Dataset — Predictive Maintenance**

Collapses each cycle's raw time-series (up to 6000 timesteps per sensor) into a flat feature row ready for XGBoost.

Feature groups built here:
- **Group A** — Time-domain statistics for all 17 sensors (mean, std, RMS, min, max, range, kurtosis, skewness, slope, argmax)
- **Group A partial** — Crest factor for 8 high-variance sensors only (PS1–PS6, EPS1, VS1)
- **Group B** — Frequency-domain features for 9 high-frequency sensors (PS1–PS6, EPS1, FS1, FS2)
- **Group C** — Cross-sensor interaction features (pressure differentials, hydraulic power, flow imbalance)

**Final output:** `data/processed/features.parquet` — shape (~1449, ~191)

## 3.0 — Imports & configuration

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from scipy import stats
from scipy.fft import fft, fftfreq
from scipy.stats import linregress
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Paths (same pattern as Phase 2) ───────────────────────────────
BASE_DIR      = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
SENSORS_DIR   = PROCESSED_DIR / 'sensors'

# ── Sensor metadata ────────────────────────────────────────────────
SENSOR_META = {
    'PS1':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'PS2':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'PS3':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'PS4':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'PS5':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'PS6':  {'unit': 'bar',   'hz': 100, 'type': 'pressure'},
    'EPS1': {'unit': 'W',     'hz': 100, 'type': 'motor_power'},
    'FS1':  {'unit': 'l/min', 'hz': 10,  'type': 'flow'},
    'FS2':  {'unit': 'l/min', 'hz': 10,  'type': 'flow'},
    'TS1':  {'unit': '°C',    'hz': 1,   'type': 'temperature'},
    'TS2':  {'unit': '°C',    'hz': 1,   'type': 'temperature'},
    'TS3':  {'unit': '°C',    'hz': 1,   'type': 'temperature'},
    'TS4':  {'unit': '°C',    'hz': 1,   'type': 'temperature'},
    'VS1':  {'unit': 'mm/s',  'hz': 1,   'type': 'vibration'},
    'CE':   {'unit': '%',     'hz': 1,   'type': 'cooling_efficiency'},
    'CP':   {'unit': 'kW',    'hz': 1,   'type': 'cooling_power'},
    'SE':   {'unit': '%',     'hz': 1,   'type': 'efficiency'},
}

ALL_SENSORS = list(SENSOR_META.keys())

# Sensors that get crest factor (high variance, meaningful peaks)
CREST_SENSORS = ['PS1', 'PS2', 'PS3', 'PS4', 'PS5', 'PS6', 'EPS1', 'VS1']

# Sensors that get FFT features (need >= 600 samples for useful frequency resolution)
FFT_SENSORS = ['PS1', 'PS2', 'PS3', 'PS4', 'PS5', 'PS6', 'EPS1', 'FS1', 'FS2']

TARGETS = ['cooler', 'valve', 'pump', 'accumulator']

print('Configuration loaded ✓')
print(f'  All sensors:    {len(ALL_SENSORS)}')
print(f'  Crest sensors:  {len(CREST_SENSORS)}')
print(f'  FFT sensors:    {len(FFT_SENSORS)}')

## 3.1 — Load cleaned sensor data from Phase 2

In [ ]:
print('Loading cleaned sensor arrays from data/processed/sensors/...\n')

sensors = {}
for name in ALL_SENSORS:
    path = SENSORS_DIR / f'{name}.parquet'
    sensors[name] = pd.read_parquet(path).values  # Load as numpy array for speed
    print(f'  {name:<6} {sensors[name].shape}')

labels_encoded = pd.read_csv(PROCESSED_DIR / 'labels_encoded.csv')
labels_raw     = pd.read_csv(PROCESSED_DIR / 'labels_raw.csv')

N_CYCLES = sensors['PS1'].shape[0]
print(f'\nLoaded {len(sensors)} sensors, {N_CYCLES} cycles each ✓')
print(f'Labels shape: {labels_encoded.shape} ✓')

## 3.2 — Group A: Time-domain statistics

For every sensor and every cycle, compute:
`mean, std, RMS, min, max, range, kurtosis, skewness, slope, argmax`

Result: 17 sensors × 10 features = **170 features**

In [ ]:
def time_domain_features(signal_2d, sensor_name):
    """
    Compute 10 time-domain statistics for every cycle in signal_2d.

    Parameters
    ----------
    signal_2d : np.ndarray, shape (n_cycles, n_timesteps)
    sensor_name : str

    Returns
    -------
    pd.DataFrame, shape (n_cycles, 10)
    """
    s = sensor_name  # shorthand for column naming
    n_cycles, n_ts = signal_2d.shape

    # Vectorised across all cycles at once using numpy axis=1
    mean    = signal_2d.mean(axis=1)
    std     = signal_2d.std(axis=1)
    rms     = np.sqrt(np.mean(signal_2d ** 2, axis=1))
    sig_min = signal_2d.min(axis=1)
    sig_max = signal_2d.max(axis=1)
    rng     = sig_max - sig_min                              # peak-to-peak range
    kurt    = stats.kurtosis(signal_2d, axis=1)              # Fisher definition (normal=0)
    skew    = stats.skew(signal_2d, axis=1)
    argmax  = signal_2d.argmax(axis=1) / n_ts               # normalised position of max (0–1)

    # Slope: fit a line to each cycle and extract gradient
    x = np.arange(n_ts)
    # Vectorised linear regression using least squares formula
    x_mean  = x.mean()
    x_var   = ((x - x_mean) ** 2).sum()
    slope   = ((signal_2d - signal_2d.mean(axis=1, keepdims=True))
               * (x - x_mean)).sum(axis=1) / x_var

    return pd.DataFrame({
        f'{s}_mean':   mean,
        f'{s}_std':    std,
        f'{s}_rms':    rms,
        f'{s}_min':    sig_min,
        f'{s}_max':    sig_max,
        f'{s}_range':  rng,
        f'{s}_kurt':   kurt,
        f'{s}_skew':   skew,
        f'{s}_slope':  slope,
        f'{s}_argmax': argmax,
    })


print('Computing Group A — time-domain statistics (17 sensors × 10 features)...\n')

group_a_parts = []
for name in ALL_SENSORS:
    df_feat = time_domain_features(sensors[name], name)
    group_a_parts.append(df_feat)
    print(f'  {name:<6} → {df_feat.shape[1]} features')

group_a = pd.concat(group_a_parts, axis=1)
print(f'\nGroup A shape: {group_a.shape}  (expected {N_CYCLES} × 170) ✓')

## 3.3 — Group A partial: Crest factor

Only for sensors with meaningful peaks: PS1–PS6, EPS1, VS1.
Crest factor = max / RMS — measures how spiky a signal is.

Result: 8 sensors × 1 feature = **8 features**

In [ ]:
def crest_factor_feature(signal_2d, sensor_name):
    """
    Crest factor = peak / RMS.
    High crest factor → impulsive signal → early-stage fault signature.
    Avoid division by zero for flat signals.
    """
    rms  = np.sqrt(np.mean(signal_2d ** 2, axis=1))
    peak = signal_2d.max(axis=1)
    cf   = np.where(rms > 1e-10, peak / rms, 0.0)
    return pd.DataFrame({f'{sensor_name}_crest': cf})


print('Computing Group A partial — crest factor (8 sensors)...\n')

group_a2_parts = []
for name in CREST_SENSORS:
    df_feat = crest_factor_feature(sensors[name], name)
    group_a2_parts.append(df_feat)
    print(f'  {name:<6} → crest factor  (mean={df_feat.iloc[:,0].mean():.3f})')

group_a2 = pd.concat(group_a2_parts, axis=1)
print(f'\nGroup A partial shape: {group_a2.shape}  (expected {N_CYCLES} × 8) ✓')

## 3.4 — Group B: Frequency-domain features

Applied to PS1–PS6 (100Hz, 6000 samples) and EPS1 (100Hz), FS1/FS2 (10Hz, 600 samples).
Skipped for all 1Hz sensors — only 60 samples gives too few FFT bins to be meaningful.

Features per sensor: spectral energy, dominant frequency, spectral centroid

Result: 9 sensors × 3 features = **27 features**

In [ ]:
def frequency_domain_features(signal_2d, sensor_name, hz):
    """
    Compute 3 frequency-domain features per cycle via FFT.

    - spectral_energy:   sum of squared FFT magnitudes (total signal power)
    - dominant_freq:     frequency (Hz) at peak FFT magnitude
    - spectral_centroid: weighted mean frequency (centre of mass of spectrum)

    Only uses the positive-frequency half of the FFT (standard practice).
    """
    n_cycles, n_ts = signal_2d.shape
    s = sensor_name

    # Frequency axis (positive half only)
    freqs = fftfreq(n_ts, d=1.0/hz)[:n_ts//2]

    spectral_energy    = np.zeros(n_cycles)
    dominant_freq      = np.zeros(n_cycles)
    spectral_centroid  = np.zeros(n_cycles)

    # Loop over cycles — FFT doesn't vectorise as cleanly as time-domain ops
    for i in range(n_cycles):
        # Apply Hanning window to reduce spectral leakage
        window    = np.hanning(n_ts)
        windowed  = signal_2d[i] * window
        fft_vals  = np.abs(fft(windowed))[:n_ts//2]

        spectral_energy[i]   = np.sum(fft_vals ** 2)
        dominant_freq[i]     = freqs[np.argmax(fft_vals)]

        # Spectral centroid: sum(freq * magnitude) / sum(magnitude)
        total_mag = fft_vals.sum()
        spectral_centroid[i] = (
            np.sum(freqs * fft_vals) / total_mag if total_mag > 1e-10 else 0.0
        )

    return pd.DataFrame({
        f'{s}_spec_energy':   spectral_energy,
        f'{s}_dom_freq':      dominant_freq,
        f'{s}_spec_centroid': spectral_centroid,
    })


print('Computing Group B — frequency-domain features (9 sensors × 3 features)...')
print('(This is the slowest step — FFT on 9 × 1449 cycles)\n')

group_b_parts = []
for name in FFT_SENSORS:
    hz      = SENSOR_META[name]['hz']
    df_feat = frequency_domain_features(sensors[name], name, hz)
    group_b_parts.append(df_feat)
    print(f'  {name:<6} ({hz:>3}Hz) → spec_energy, dom_freq, spec_centroid ✓')

group_b = pd.concat(group_b_parts, axis=1)
print(f'\nGroup B shape: {group_b.shape}  (expected {N_CYCLES} × 27) ✓')

## 3.5 — Group C: Cross-sensor interaction features

Physically-grounded domain features — the own-contribution additions:

| Feature | Formula | Detects |
|---|---|---|
| Pressure drop 1→2 | mean(PS1) − mean(PS2) | Valve restriction / blockage |
| Pressure drop 2→4 | mean(PS2) − mean(PS4) | Downstream restriction |
| Hydraulic power | mean(PS1) × mean(FS1) | Pump efficiency degradation |
| Power efficiency ratio | hydraulic power / mean(EPS1) | Electro-hydraulic efficiency |
| Flow imbalance | mean(FS1) − mean(FS2) | Leak between circuits |
| Thermal load per watt | mean(TS1) / mean(EPS1) | Cooler degradation |

Result: **6 features**

In [ ]:
print('Computing Group C — cross-sensor interaction features...\n')

# Pre-extract cycle means for sensors used in interaction features
ps1_mean  = sensors['PS1'].mean(axis=1)
ps2_mean  = sensors['PS2'].mean(axis=1)
ps4_mean  = sensors['PS4'].mean(axis=1)
fs1_mean  = sensors['FS1'].mean(axis=1)
fs2_mean  = sensors['FS2'].mean(axis=1)
eps1_mean = sensors['EPS1'].mean(axis=1)
ts1_mean  = sensors['TS1'].mean(axis=1)

# ── Pressure differentials ─────────────────────────────────────────
# Pressure drop across the valve (PS1 upstream → PS2 downstream)
# A degraded valve shows an abnormal pressure drop
pressure_drop_1_2 = ps1_mean - ps2_mean

# Pressure drop in the secondary circuit (PS2 → PS4)
# Detects downstream restrictions or accumulator issues
pressure_drop_2_4 = ps2_mean - ps4_mean

# ── Hydraulic power ────────────────────────────────────────────────
# Hydraulic power (W) = pressure (Pa) × flow (m³/s)
# Here: bar × l/min gives a proportional metric (not in SI units, but consistent)
# A pump with internal leakage delivers less hydraulic power for the same electrical input
hydraulic_power = ps1_mean * fs1_mean

# Power efficiency ratio: hydraulic power delivered vs electrical power consumed
# Drops as the pump degrades
power_efficiency = np.where(
    eps1_mean > 1e-6,
    hydraulic_power / eps1_mean,
    0.0
)

# ── Flow imbalance ─────────────────────────────────────────────────
# FS1 = primary working circuit, FS2 = cooling/filtration circuit
# In a healthy system these maintain a consistent relationship
# A leak between circuits creates an abnormal flow imbalance
flow_imbalance = fs1_mean - fs2_mean

# ── Thermal load per watt ──────────────────────────────────────────
# Temperature relative to electrical power input
# A degraded cooler lets temperature rise relative to power input
thermal_load_per_watt = np.where(
    eps1_mean > 1e-6,
    ts1_mean / eps1_mean,
    0.0
)

group_c = pd.DataFrame({
    'cross_pressure_drop_1_2':    pressure_drop_1_2,
    'cross_pressure_drop_2_4':    pressure_drop_2_4,
    'cross_hydraulic_power':      hydraulic_power,
    'cross_power_efficiency':     power_efficiency,
    'cross_flow_imbalance':       flow_imbalance,
    'cross_thermal_load_per_watt': thermal_load_per_watt,
})

for col in group_c.columns:
    print(f'  {col:<35}  mean={group_c[col].mean():>10.4f}  std={group_c[col].std():>9.4f}')

print(f'\nGroup C shape: {group_c.shape}  (expected {N_CYCLES} × 6) ✓')

## 3.6 — Assemble final feature matrix

In [ ]:
features = pd.concat([
    group_a,   # 170 features — time domain all sensors
    group_a2,  #   8 features — crest factor
    group_b,   #  27 features — frequency domain
    group_c,   #   6 features — cross-sensor
], axis=1)

# Reset index to ensure alignment with labels
features = features.reset_index(drop=True)

print('Feature matrix assembled\n')
print(f'  Shape:          {features.shape}')
print(f'  Group A:        {group_a.shape[1]} features  (time-domain all sensors)')
print(f'  Group A crest:  {group_a2.shape[1]} features  (crest factor)')
print(f'  Group B:        {group_b.shape[1]} features  (frequency domain)')
print(f'  Group C:        {group_c.shape[1]} features  (cross-sensor)')
print(f'  Total:          {features.shape[1]} features')
print()
print(f'  NaN count:      {features.isna().sum().sum()}')
print(f'  Inf count:      {np.isinf(features.values).sum()}')

## 3.7 — Handle any NaN / Inf values

In [ ]:
# Replace any Inf values with NaN first, then fill
features.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_counts = features.isna().sum()
problem_cols = nan_counts[nan_counts > 0]

if len(problem_cols) > 0:
    print(f'Found {len(problem_cols)} columns with NaN/Inf — filling with column median:\n')
    for col, count in problem_cols.items():
        median_val = features[col].median()
        features[col].fillna(median_val, inplace=True)
        print(f'  {col}: {count} NaN → filled with median={median_val:.4f}')
else:
    print('No NaN or Inf values found ✓')

# Final check
assert features.isna().sum().sum() == 0,       'NaN values remain after cleaning'
assert not np.isinf(features.values).any(),    'Inf values remain after cleaning'
assert features.shape[0] == labels_encoded.shape[0], 'Row count mismatch with labels'

print(f'\nFinal feature matrix: {features.shape} — clean and aligned with labels ✓')

## 3.8 — Save

In [ ]:
# Save features
features_path = PROCESSED_DIR / 'features.parquet'
features.to_parquet(features_path, index=False)

# Save feature group metadata (useful for Phase 5 SHAP analysis)
import json
feature_groups = {
    'group_a_time_domain':   [c for c in features.columns if any(
        c.endswith(f'_{s}') for s in ['mean','std','rms','min','max','range','kurt','skew','slope','argmax']
    )],
    'group_a_crest':         [c for c in features.columns if c.endswith('_crest')],
    'group_b_frequency':     [c for c in features.columns if any(
        s in c for s in ['spec_energy','dom_freq','spec_centroid']
    )],
    'group_c_cross_sensor':  [c for c in features.columns if c.startswith('cross_')],
}

with open(PROCESSED_DIR / 'feature_groups.json', 'w') as f:
    json.dump(feature_groups, f, indent=2)

# Save column list
with open(PROCESSED_DIR / 'feature_columns.json', 'w') as f:
    json.dump(list(features.columns), f, indent=2)

print('Saved to data/processed/:\n')
print(f'  features.parquet       {features.shape}')
print(f'  feature_groups.json    {len(feature_groups)} groups')
print(f'  feature_columns.json   {len(features.columns)} column names')

## 3.9 — Diagnostic plots

In [ ]:
# ── Plot 1: Feature group breakdown (bar chart) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Left: feature count by group
ax = axes[0]
ax.set_facecolor('#161b22')
group_names  = ['Time-domain\n(all sensors)', 'Crest factor\n(8 sensors)',
                'Frequency\ndomain', 'Cross-sensor\ninteractions']
group_counts = [group_a.shape[1], group_a2.shape[1], group_b.shape[1], group_c.shape[1]]
group_colors = ['#7F77DD', '#1D9E75', '#EF9F27', '#D85A30']

bars = ax.bar(group_names, group_counts, color=group_colors, alpha=0.85,
              edgecolor='#1a1a2e', linewidth=0.5)
for bar, count in zip(bars, group_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', color='white', fontsize=11, fontweight='bold')
ax.set_title(f'Feature count by group  (total={features.shape[1]})',
             color='white', fontsize=11, pad=8)
ax.set_ylabel('Number of features', color='#888', fontsize=9)
ax.tick_params(colors='#888', labelsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#30363d')

# Right: feature variance distribution (log scale) — shows spread is reasonable
ax2 = axes[1]
ax2.set_facecolor('#161b22')
variances = np.log1p(features.var())
ax2.hist(variances, bins=40, color='#7F77DD', alpha=0.85,
         edgecolor='#1a1a2e', linewidth=0.3)
ax2.set_title('Feature variance distribution (log scale)\n— check for near-zero variance features',
              color='white', fontsize=11, pad=8)
ax2.set_xlabel('log(1 + variance)', color='#888', fontsize=9)
ax2.set_ylabel('Number of features', color='#888', fontsize=9)
ax2.tick_params(colors='#888', labelsize=9)
for spine in ax2.spines.values():
    spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_feature_groups.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Plot 2: Key feature distributions split by target class ────────
# Shows whether features actually separate the classes — visual validation
# Pick the most physically meaningful feature per target

showcase = [
    ('TS1_mean',                 'cooler',      'Cooler: TS1 mean temperature'),
    ('PS1_std',                  'valve',       'Valve: PS1 pressure std dev'),
    ('cross_flow_imbalance',     'pump',        'Pump: flow imbalance (FS1−FS2)'),
    ('cross_pressure_drop_2_4',  'accumulator', 'Accumulator: pressure drop PS2−PS4'),
]

LABEL_MAPS = {
    'cooler':      {0: 'near failure', 1: 'reduced eff.', 2: 'full eff.'},
    'valve':       {0: 'near failure', 1: 'severe lag',   2: 'small lag', 3: 'optimal'},
    'pump':        {0: 'no leakage',   1: 'weak leakage', 2: 'severe leakage'},
    'accumulator': {0: 'near failure', 1: 'severely red.',2: 'slightly red.', 3: 'optimal'},
}

palette = ['#E24B4A', '#EF9F27', '#1D9E75', '#7F77DD']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for ax, (feat, target, title) in zip(axes, showcase):
    ax.set_facecolor('#161b22')
    classes = sorted(labels_encoded[target].unique())

    for cls, color in zip(classes, palette):
        mask   = labels_encoded[target] == cls
        values = features.loc[mask, feat].values
        label  = LABEL_MAPS[target][cls]
        ax.hist(values, bins=30, alpha=0.6, color=color,
                label=f'Class {cls}: {label}',
                edgecolor='#1a1a2e', linewidth=0.3)

    ax.set_title(title, color='white', fontsize=10, pad=6)
    ax.set_xlabel(feat, color='#888', fontsize=8)
    ax.set_ylabel('Cycles', color='#888', fontsize=8)
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.legend(fontsize=7, labelcolor='white', facecolor='#161b22',
              edgecolor='#30363d', framealpha=0.8)

fig.suptitle('Class separation per key feature — do distributions differ by health state?',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_feature_class_separation.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Plot 3: Correlation heatmap of cross-sensor features ───────────
# Quick check that Group C features aren't perfectly collinear

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

corr = group_c.corr()
short_names = [c.replace('cross_','') for c in group_c.columns]

im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(short_names)))
ax.set_yticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=35, ha='right', color='#888', fontsize=8)
ax.set_yticklabels(short_names, color='#888', fontsize=8)

# Annotate each cell
for i in range(len(short_names)):
    for j in range(len(short_names)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}',
                ha='center', va='center', fontsize=7,
                color='white' if abs(corr.values[i,j]) > 0.5 else '#888')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Group C cross-sensor feature correlations',
             color='white', fontsize=11, pad=10)
for spine in ax.spines.values():
    spine.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_crosssensor_correlation.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Quick feature sanity summary ───────────────────────────────────
print('Feature matrix summary\n')
print(f'  Total features:      {features.shape[1]}')
print(f'  Total cycles:        {features.shape[0]}')
print(f'  NaN:                 {features.isna().sum().sum()}')
print(f'  Inf:                 {np.isinf(features.values).sum()}')

# Flag near-zero variance features (potential issues for some models)
low_var = (features.var() < 1e-6).sum()
print(f'  Near-zero variance:  {low_var} features')
if low_var > 0:
    print('  ⚠  Consider dropping near-zero variance features before training')
else:
    print('  ✓  No near-zero variance features')

print()
print('Feature group breakdown:')
print(f'  Group A  time-domain:    {group_a.shape[1]:>4} features  (17 sensors × 10 stats)')
print(f'  Group A  crest factor:   {group_a2.shape[1]:>4} features  (8 sensors × 1)')
print(f'  Group B  frequency:      {group_b.shape[1]:>4} features  (9 sensors × 3)')
print(f'  Group C  cross-sensor:   {group_c.shape[1]:>4} features  (6 interactions)')
print(f'           TOTAL:          {features.shape[1]:>4} features')

print()
print('Sample of feature names:')
for col in features.columns[:5]:
    print(f'  {col}')
print('  ...')
for col in features.columns[-5:]:
    print(f'  {col}')

## 3.10 — Summary

| Group | Sensors | Features | Rationale |
|---|---|---|---|
| A — Time domain | All 17 | 170 | Core statistical fingerprint per cycle |
| A — Crest factor | PS1–PS6, EPS1, VS1 | 8 | Impulsive fault detection |
| B — Frequency domain | PS1–PS6, EPS1, FS1, FS2 | 27 | Spectral fault signatures |
| C — Cross-sensor | PS1–PS4, FS1, FS2, EPS1, TS1 | 6 | Domain-informed interaction features |
| **Total** | | **211** | |

**Next → `04_modelling.ipynb`**

Train XGBoost multi-output classifier. Compare with random forest and logistic regression baseline. Evaluate with stratified 5-fold cross-validation and macro F1.